# 입출력과 출처 검증

모델 호출 앞뒤에서 질문과 출처 형식을 파이썬으로 검사합니다.

In [1]:
%pip install -q langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 실행 환경 준비

LangChain 1.4와 OpenAI 통합 버전을 고정하고 실제 API를 호출합니다. 예상 비용은 0.01달러 미만입니다.

In [2]:
from pathlib import Path
import os, sys
from dotenv import find_dotenv, load_dotenv

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part4/ch16-guardrails").is_dir())
LAB = ROOT / "labs/part4/ch16-guardrails"
load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "프로젝트 루트 .env에 OPENAI_API_KEY가 필요합니다."
print("Python:", sys.version.split()[0], "| API 키 준비: True")

Python: 3.12.13 | API 키 준비: True


In [3]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4-mini", temperature=0, max_tokens=140,
    timeout=30, max_retries=0,
)
print("모델 준비:", type(model).__name__)

모델 준비: ChatOpenAI


## 기본 실행

Ch 9의 `[doc_id p.쪽]` 형식을 그대로 사용합니다.

In [4]:
import re

def validate_query(query: str, max_chars: int = 500) -> str:
    normalized = " ".join(query.split())
    if not normalized or len(normalized) > max_chars:
        raise ValueError("질문의 내용과 길이를 확인하세요.")
    return normalized

def validate_answer(answer: str, allowed: set[tuple[str, int]]) -> list[str]:
    citations = [(doc, int(page)) for doc, page in re.findall(r"\[([A-Z]\d{2}) p\.(\d+)\]", answer)]
    if not citations or any(item not in allowed for item in citations):
        raise ValueError("출처 허용 목록을 확인하세요.")
    return [f"[{doc} p.{page}]" for doc, page in citations]

question = validate_query("  도서관 휴관일은 언제인가요?  ")
context = "E01 1쪽: 미르시립도서관은 매주 월요일에 휴관합니다."
messages = [
    {"role": "system", "content": "문서만 사용하고 끝에 [E01 p.1]을 적으세요."},
    {"role": "user", "content": f"질문: {question}\n문서: {context}"},
]
answer = model.invoke(messages)
print(answer.text)
print("검증 출처:", validate_answer(answer.text, {("E01", 1)}))

미르시립도서관은 매주 월요일에 휴관합니다. [E01 p.1]
검증 출처: ['[E01 p.1]']


## 중간 결과 관찰

질문을 정규화하고 허용한 문서 ID와 쪽만 통과시킵니다.

In [5]:
print("질문 글자 수:", len(question))
print("AIMessage.text 있음:", bool(answer.text.strip()))
assert "[E01 p.1]" in answer.text

질문 글자 수: 15
AIMessage.text 있음: True


## 값 하나를 바꿔 비교

검색 결과에 없는 출처를 넣으면 후처리에서 거부됩니다.

In [6]:
# 검색 결과에 없는 E02는 모델 호출 뒤에도 허용하지 않습니다.
try:
    validate_answer("화요일입니다. [E02 p.1]", {("E01", 1)})
except ValueError as error:
    print(type(error).__name__, error)

ValueError 출처 허용 목록을 확인하세요.


## 수정

모델 답의 출처를 검색 결과의 허용 목록과 대조합니다.

In [7]:
citations = validate_answer(answer.text, {("E01", 1)})
assert citations == ["[E01 p.1]"]
print("허용 목록 검증 통과")

허용 목록 검증 통과
